## Importando os dados preparados

In [1]:
import joblib

partitions, anomalyColumns = joblib.load("./dados_preparados.pkl")

## Importando os pipelines

In [2]:
pipelines = joblib.load("./pipelines.pkl")
print(list(pipelines))

['IF', 'OCSVM', 'LOF']


## Imports

In [3]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from anomaly_utils import evaluateCandidates, evaluateQ95, scoreSummary, selectCandidate, transition

# Treinamento dos modelos

## One-Class SVM

Cada candidato de `nu` é ajustado somente com `normalTrain`. `nu` é um limite teórico superior
para erros de treino e inferior para vetores de suporte; as frações observadas são medidas.
O q95 usa exclusivamente as True de validação. Fake-validation só participa da escolha de `nu`
(ROC-AUC → AP → menor `nu`); nenhuma métrica de teste existe quando o candidato é escolhido.

In [4]:
def describeOneClassSvm(pipeline):
    detector = pipeline["detector"]
    trainingFeatures = partitions["normalTrain"][anomalyColumns]
    return {
        "nu": detector.nu,
        "resolvedGamma": float(detector._gamma),
        "supportVectorCount": int(detector.support_.size),
        "supportVectorFraction": float(detector.support_.size / len(trainingFeatures)),
        "normalTrainNativeOutlierFraction": float(np.mean(pipeline.predict(trainingFeatures) == -1)),
    }


fittedPipelines, validationResults = evaluateCandidates(
    pipelines["OCSVM"], partitions, anomalyColumns, describe=describeOneClassSvm,
)
selectedModelKey = selectCandidate(validationResults, "nu")
display(validationResults.drop(columns=["nativeMetrics", "q95Metrics"]))
print("Candidato congelado antes do teste:", selectedModelKey)

,modelKey,nu,resolvedGamma,supportVectorCount,supportVectorFraction,normalTrainNativeOutlierFraction,fitSeconds,q95Threshold,normalValidationQ95AlertRate,validationRocAuc,validationAveragePrecision
0,ocsvm_nu_001,0.010,0.2,61,0.028241,0.016204,0.032855,-0.000891,0.05,0.583956,0.820604
1,ocsvm_nu_0025,0.025,0.2,77,0.035648,0.029630,0.022586,-0.002486,0.05,0.730942,0.895087
2,ocsvm_nu_005,0.050,0.2,124,0.057407,0.050463,0.020197,0.215298,0.05,0.815887,0.927092
3,ocsvm_nu_010,0.100,0.2,225,0.104167,0.103241,0.031076,3.666521,0.05,0.969813,0.979765


Candidato congelado antes do teste: ocsvm_nu_010


## Avaliação no teste e comparação com o Isolation Forest

O candidato escolhido e o Isolation Forest usam o mesmo treino, features e regra q95.
A fronteira nativa do OCSVM (`predict == -1`) é reportada separada do q95.

In [5]:
ocsvm = evaluateQ95(fittedPipelines[selectedModelKey], partitions, anomalyColumns, fit=False)
isolationForest = evaluateQ95(pipelines["IF"], partitions, anomalyColumns)

testPredictionsFrame = ocsvm["testFrame"][["id", "label"] + anomalyColumns].copy()
testPredictionsFrame["ocsvmAnomalyScore"] = ocsvm["scores"]
testPredictionsFrame["isolationForestAnomalyScore"] = isolationForest["scores"]
testPredictionsFrame["transition"] = transition(ocsvm["q95Flags"], isolationForest["q95Flags"], "ocsvm", "isolationForest")

display(pd.concat({
    "ocsvm": scoreSummary(testPredictionsFrame, "ocsvmAnomalyScore"),
    "isolationForest": scoreSummary(testPredictionsFrame, "isolationForestAnomalyScore"),
}, names=["model", "label"]))
display(pd.DataFrame({
    "OCSVM nativo": ocsvm["nativeMetrics"],
    "OCSVM q95": ocsvm["q95Metrics"],
    "Isolation Forest q95": isolationForest["q95Metrics"],
}).T)
testPredictionsFrame.groupby(["label", "transition"]).size().rename("count").reset_index()

class  count       mean     median       std       min  \
model           label                                                          
ocsvm           0      True    720  -1.148508  -1.587920  2.789851 -3.769342   
                1      Fake   1800  10.892979  10.240216  5.719358 -3.779833   
isolationForest 0      True    720  -0.058204  -0.076152  0.061314 -0.125163   
                1      Fake   1800   0.148692   0.142426  0.037844 -0.124589   

                             max  
model           label             
ocsvm           0      21.352744  
                1      23.014737  
isolationForest 0       0.212721  
                1       0.261826

,rocAuc,averagePrecision,fakePrevalence,tn,fp,fn,tp,accuracy,precision,recall,f1,fpr
OCSVM nativo,0.976384,0.985442,0.714286,650.0,70.0,25.0,1775.0,0.962302,0.962060,0.986111,0.973937,0.097222
OCSVM q95,0.976384,0.985442,0.714286,693.0,27.0,145.0,1655.0,0.931746,0.983948,0.919444,0.950603,0.037500
Isolation Forest q95,0.980673,0.988672,0.714286,699.0,21.0,31.0,1769.0,0.979365,0.988268,0.982778,0.985515,0.029167


,label,transition,count
0,0,alerta_adicionado_ocsvm,6
1,0,alerta_mantido,21
2,0,sem_alerta,693
3,1,alerta_adicionado_isolationForest,114
4,1,alerta_mantido,1655
5,1,sem_alerta,31


## Conclusão

In [6]:
ocsvmQ95, isolationForestQ95 = ocsvm["q95Metrics"], isolationForest["q95Metrics"]
display(Markdown(f"""
O candidato congelado foi **{selectedModelKey}**. No teste, OCSVM q95 obteve
ROC-AUC **{ocsvmQ95['rocAuc']:.4f}**, AP **{ocsvmQ95['averagePrecision']:.4f}**,
recall **{ocsvmQ95['recall']:.2%}** e FPR **{ocsvmQ95['fpr']:.2%}**.
O Isolation Forest q95 obteve ROC-AUC **{isolationForestQ95['rocAuc']:.4f}**,
AP **{isolationForestQ95['averagePrecision']:.4f}**, recall **{isolationForestQ95['recall']:.2%}**
e FPR **{isolationForestQ95['fpr']:.2%}**.

Estes números descrevem desvio linguístico neste split; não provam falsidade nem
superioridade estatística. A seleção de `nu` foi assistida por labels Fake de validação.
"""))


O candidato congelado foi **ocsvm_nu_010**. No teste, OCSVM q95 obteve
ROC-AUC **0.9764**, AP **0.9854**,
recall **91.94%** e FPR **3.75%**.
O Isolation Forest q95 obteve ROC-AUC **0.9807**,
AP **0.9887**, recall **98.28%**
e FPR **2.92%**.

Estes números descrevem desvio linguístico neste split; não provam falsidade nem
superioridade estatística. A seleção de `nu` foi assistida por labels Fake de validação.
